In [92]:
# Load general utilities
# ----------------------
import pandas as pd
import matplotlib.pyplot as plt
import datetime
import numpy as np
import pickle
import time
import seaborn as sns
import os

# Load sklearn utilities
# ----------------------
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn import preprocessing
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error

from sklearn.metrics import accuracy_score, precision_score, classification_report,recall_score, roc_auc_score, roc_curve, brier_score_loss, f1_score,mean_squared_error, r2_score

from sklearn.calibration import calibration_curve

# Load classifiers
# ----------------
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import RidgeClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegressionCV

# Load debugger, if required
#import pixiedust
pd.options.mode.chained_assignment = None #'warn'

In [93]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
import scikitplot as skplt
from sklearn import metrics   

In [94]:
pd.options.display.max_columns = None

In [95]:
import warnings
warnings.filterwarnings("ignore")

## Importing and preparing data

In [96]:
df = pd.read_csv("creditcard.csv")

In [97]:
df

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,-0.551600,-0.617801,-0.991390,-0.311169,1.468177,-0.470401,0.207971,0.025791,0.403993,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,1.612727,1.065235,0.489095,-0.143772,0.635558,0.463917,-0.114805,-0.183361,-0.145783,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,0.624501,0.066084,0.717293,-0.165946,2.345865,-2.890083,1.109969,-0.121359,-2.261857,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,-0.226487,0.178228,0.507757,-0.287924,-0.631418,-1.059647,-0.684093,1.965775,-1.232622,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,-0.822843,0.538196,1.345852,-1.119670,0.175121,-0.451449,-0.237033,-0.038195,0.803487,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,172786.0,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,4.356170,-1.593105,2.711941,-0.689256,4.626942,-0.924459,1.107641,1.991691,0.510632,-0.682920,1.475829,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,0.77,0
284803,172787.0,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,-0.975926,-0.150189,0.915802,1.214756,-0.675143,1.164931,-0.711757,-0.025693,-1.221179,-1.545556,0.059616,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,24.79,0
284804,172788.0,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,-0.484782,0.411614,0.063119,-0.183699,-0.510602,1.329284,0.140716,0.313502,0.395652,-0.577252,0.001396,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,67.88,0
284805,172788.0,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,-0.399126,-1.933849,-0.962886,-1.042082,0.449624,1.962563,-0.608577,0.509928,1.113981,2.897849,0.127434,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,10.00,0


In [98]:
df.shape

(284807, 31)

In [99]:
df['Class'].value_counts()

0    284315
1       492
Name: Class, dtype: int64

In [100]:
df = df.drop_duplicates()

In [101]:
df = df.dropna()    # Dropping the missing values.


## Handling Outliers

In [102]:
n_rows = len(df)
df = df[df.Amount < 500]
print("Removed " + str(n_rows - len(df)) + " rows")

Removed 9459 rows


In [103]:
fraud=df[df['Class']==1]
non_fraud=df[df['Class']==0]

In [104]:
fraud.shape

(439, 31)

In [105]:
non_fraud.shape

(273828, 31)

In [106]:
y = df["Class"]
X = df.drop("Class", axis=1)

## Train-Test Split

In [107]:
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=886)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [108]:
X_train.shape, X_test.shape

((219413, 30), (54854, 30))

In [109]:
X_train

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
3003,2553.0,1.026734,-0.426098,1.156206,1.419335,-1.275691,-0.236011,-0.652152,0.223332,1.280394,-0.154982,-1.239793,-0.907966,-2.569829,0.175171,0.473826,0.210629,-0.098547,0.273263,-0.212422,-0.170105,-0.053759,-0.222542,-0.030280,0.322155,0.303491,-0.407033,0.043183,0.043897,73.85
274864,166250.0,-0.563415,0.802934,-1.669090,-0.261398,2.957650,-1.795139,1.508199,-0.661698,-0.559874,-1.386034,-0.264703,-0.378463,0.546684,-2.899568,-0.526601,0.258890,1.781098,0.685203,-0.354173,0.110445,0.039278,0.330404,-0.305221,0.330981,-0.209888,0.511143,-0.196001,0.150965,0.76
67737,52675.0,-0.937891,0.521930,0.948435,0.029805,2.379072,4.482743,0.199403,0.501896,0.499023,0.333833,-0.705018,-0.201732,-0.341244,-0.896365,-0.421899,-0.768539,-0.275010,0.169852,1.401466,0.079910,-0.191924,-0.111662,-0.576502,1.045512,0.554622,-0.242230,-0.671419,-0.408828,49.00
158419,111203.0,-0.449106,0.764294,0.719073,-0.991448,0.618617,0.150455,0.778490,0.038102,0.969926,-0.826269,1.152264,-2.989862,0.041851,2.122635,-0.721263,0.819101,-0.299862,0.536965,0.258230,-0.198491,-0.516980,-1.542562,0.187167,0.038878,-0.856940,-0.079577,-0.087211,0.135022,58.63
1911,1472.0,-4.422952,-4.841034,2.071464,1.681590,5.036754,-3.047061,-2.226676,0.447233,-0.807602,0.470984,1.224514,0.504005,-0.465582,0.416172,-0.363002,0.860582,-0.857442,-0.435671,-1.456260,1.514762,0.248325,-1.109773,1.366868,-0.068259,-0.374771,0.330559,-0.387578,-0.009215,140.55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
213775,139389.0,-0.876148,1.045461,1.273240,1.657455,-1.067039,1.262843,0.582252,-0.092462,-0.133993,1.008125,0.896476,0.433083,0.749491,-0.131165,1.886321,-0.122431,-0.425483,1.277919,2.274461,-0.025376,0.150437,0.838878,-0.098471,-0.385105,-0.897054,-0.470444,-0.710263,-0.306786,175.75
212706,138966.0,-0.819166,1.457983,-0.436527,-0.498289,-0.278575,-0.700525,-0.587829,-2.282368,-0.229840,-1.082471,-1.423677,-0.291712,-1.724043,1.146803,-0.351999,0.268071,-0.009790,-0.476827,-0.246436,0.378021,-1.491904,-0.307902,0.369514,-0.024796,-0.588745,0.128915,0.080042,0.065366,9.98
272653,165199.0,-0.266453,0.571375,0.562362,0.089295,0.492446,0.170931,0.021580,0.157566,0.419499,-0.788425,-1.267510,-0.890194,-0.602117,-1.351533,0.850950,0.309796,0.677937,1.008036,0.695394,0.024777,0.217305,0.750526,-0.123508,0.224045,-0.672729,0.700434,0.059633,0.235761,7.10
115367,73847.0,1.122975,-0.203277,0.979444,0.544087,-0.873768,-0.205654,-0.566887,0.140287,0.597574,-0.138286,-0.232565,-0.087735,-0.520864,0.148299,1.733383,0.401046,-0.252023,-0.356488,-0.736505,-0.098587,0.032741,0.050305,0.075314,0.080022,0.044136,0.420972,0.000061,0.026389,33.96


In [110]:
y_train.shape, y_test.shape

((219413,), (54854,))

In [111]:
y_train.value_counts()

0    219064
1       349
Name: Class, dtype: int64

In [112]:
y_train.value_counts()/len(y_train)

0    0.998409
1    0.001591
Name: Class, dtype: float64

In [113]:
y_test.value_counts()/len(y_test)

0    0.998359
1    0.001641
Name: Class, dtype: float64

## Standardization

In [140]:
X_train

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
3003,2553.0,1.026734,-0.426098,1.156206,1.419335,-1.275691,-0.236011,-0.652152,0.223332,1.280394,-0.154982,-1.239793,-0.907966,-2.569829,0.175171,0.473826,0.210629,-0.098547,0.273263,-0.212422,-0.170105,-0.053759,-0.222542,-0.030280,0.322155,0.303491,-0.407033,0.043183,0.043897,73.85
274864,166250.0,-0.563415,0.802934,-1.669090,-0.261398,2.957650,-1.795139,1.508199,-0.661698,-0.559874,-1.386034,-0.264703,-0.378463,0.546684,-2.899568,-0.526601,0.258890,1.781098,0.685203,-0.354173,0.110445,0.039278,0.330404,-0.305221,0.330981,-0.209888,0.511143,-0.196001,0.150965,0.76
67737,52675.0,-0.937891,0.521930,0.948435,0.029805,2.379072,4.482743,0.199403,0.501896,0.499023,0.333833,-0.705018,-0.201732,-0.341244,-0.896365,-0.421899,-0.768539,-0.275010,0.169852,1.401466,0.079910,-0.191924,-0.111662,-0.576502,1.045512,0.554622,-0.242230,-0.671419,-0.408828,49.00
158419,111203.0,-0.449106,0.764294,0.719073,-0.991448,0.618617,0.150455,0.778490,0.038102,0.969926,-0.826269,1.152264,-2.989862,0.041851,2.122635,-0.721263,0.819101,-0.299862,0.536965,0.258230,-0.198491,-0.516980,-1.542562,0.187167,0.038878,-0.856940,-0.079577,-0.087211,0.135022,58.63
1911,1472.0,-4.422952,-4.841034,2.071464,1.681590,5.036754,-3.047061,-2.226676,0.447233,-0.807602,0.470984,1.224514,0.504005,-0.465582,0.416172,-0.363002,0.860582,-0.857442,-0.435671,-1.456260,1.514762,0.248325,-1.109773,1.366868,-0.068259,-0.374771,0.330559,-0.387578,-0.009215,140.55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
213775,139389.0,-0.876148,1.045461,1.273240,1.657455,-1.067039,1.262843,0.582252,-0.092462,-0.133993,1.008125,0.896476,0.433083,0.749491,-0.131165,1.886321,-0.122431,-0.425483,1.277919,2.274461,-0.025376,0.150437,0.838878,-0.098471,-0.385105,-0.897054,-0.470444,-0.710263,-0.306786,175.75
212706,138966.0,-0.819166,1.457983,-0.436527,-0.498289,-0.278575,-0.700525,-0.587829,-2.282368,-0.229840,-1.082471,-1.423677,-0.291712,-1.724043,1.146803,-0.351999,0.268071,-0.009790,-0.476827,-0.246436,0.378021,-1.491904,-0.307902,0.369514,-0.024796,-0.588745,0.128915,0.080042,0.065366,9.98
272653,165199.0,-0.266453,0.571375,0.562362,0.089295,0.492446,0.170931,0.021580,0.157566,0.419499,-0.788425,-1.267510,-0.890194,-0.602117,-1.351533,0.850950,0.309796,0.677937,1.008036,0.695394,0.024777,0.217305,0.750526,-0.123508,0.224045,-0.672729,0.700434,0.059633,0.235761,7.10
115367,73847.0,1.122975,-0.203277,0.979444,0.544087,-0.873768,-0.205654,-0.566887,0.140287,0.597574,-0.138286,-0.232565,-0.087735,-0.520864,0.148299,1.733383,0.401046,-0.252023,-0.356488,-0.736505,-0.098587,0.032741,0.050305,0.075314,0.080022,0.044136,0.420972,0.000061,0.026389,33.96


In [141]:
#from sklearn.preprocessing import StandardScaler
#Scaler_X = StandardScaler()
#X_train = Scaler_X.fit_transform(X_train)
#X_test = Scaler_X.transform(X_test)

## Oversampling method - SMOTE

In [142]:
from imblearn.over_sampling import SMOTE
from collections import Counter

counter = Counter(y_train)
print('Before',counter)
# oversampling the train dataset using SMOTE
smt = SMOTE()
X_train_sm, y_train_sm = smt.fit_resample(X_train, y_train)

counter = Counter(y_train_sm)
print('After',counter)

Before Counter({0: 219064, 1: 349})
After Counter({0: 219064, 1: 219064})


In [153]:
X_train_sm

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
0,2553.000000,1.026734,-0.426098,1.156206,1.419335,-1.275691,-0.236011,-0.652152,0.223332,1.280394,-0.154982,-1.239793,-0.907966,-2.569829,0.175171,0.473826,0.210629,-0.098547,0.273263,-0.212422,-0.170105,-0.053759,-0.222542,-0.030280,0.322155,0.303491,-0.407033,0.043183,0.043897,73.850000
1,166250.000000,-0.563415,0.802934,-1.669090,-0.261398,2.957650,-1.795139,1.508199,-0.661698,-0.559874,-1.386034,-0.264703,-0.378463,0.546684,-2.899568,-0.526601,0.258890,1.781098,0.685203,-0.354173,0.110445,0.039278,0.330404,-0.305221,0.330981,-0.209888,0.511143,-0.196001,0.150965,0.760000
2,52675.000000,-0.937891,0.521930,0.948435,0.029805,2.379072,4.482743,0.199403,0.501896,0.499023,0.333833,-0.705018,-0.201732,-0.341244,-0.896365,-0.421899,-0.768539,-0.275010,0.169852,1.401466,0.079910,-0.191924,-0.111662,-0.576502,1.045512,0.554622,-0.242230,-0.671419,-0.408828,49.000000
3,111203.000000,-0.449106,0.764294,0.719073,-0.991448,0.618617,0.150455,0.778490,0.038102,0.969926,-0.826269,1.152264,-2.989862,0.041851,2.122635,-0.721263,0.819101,-0.299862,0.536965,0.258230,-0.198491,-0.516980,-1.542562,0.187167,0.038878,-0.856940,-0.079577,-0.087211,0.135022,58.630000
4,1472.000000,-4.422952,-4.841034,2.071464,1.681590,5.036754,-3.047061,-2.226676,0.447233,-0.807602,0.470984,1.224514,0.504005,-0.465582,0.416172,-0.363002,0.860582,-0.857442,-0.435671,-1.456260,1.514762,0.248325,-1.109773,1.366868,-0.068259,-0.374771,0.330559,-0.387578,-0.009215,140.550000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
438123,40530.028860,-1.259118,0.733468,-5.913684,6.064482,2.051782,-2.538172,-1.234907,0.410892,-2.155187,-2.698549,4.788540,-3.112634,-0.723396,-8.760207,0.347597,-0.449722,1.882877,1.285725,-0.224034,-0.449082,-0.000446,0.281389,1.762501,-0.402504,-0.074995,0.288029,0.369904,0.410622,1.000000
438124,91484.942361,-0.067021,2.332816,-2.552571,1.093934,3.134361,-1.658104,1.891842,-0.909383,0.914142,-3.407343,2.062909,-4.029719,1.007560,-5.853496,-1.201913,1.515764,5.204745,2.672718,-2.127565,0.015315,-0.545223,-0.828247,-0.400109,-0.027141,0.574212,-0.708249,-0.415002,-0.344952,2.508423
438125,128963.233180,-0.953017,2.763856,-5.047006,3.530141,-0.494541,-0.787025,-1.761255,1.321456,-1.818336,-4.525364,3.217948,-3.599448,-0.973401,-7.828157,-1.819882,-1.388346,-1.909834,0.390262,-0.633158,0.040481,0.544053,0.386901,-0.054751,0.254958,0.138996,-0.190080,0.526934,0.211385,110.495034
438126,155572.926988,-0.077179,3.209722,-5.439598,5.805078,-0.114368,-2.107039,-2.628921,0.732908,-3.461498,-4.738698,3.077389,-5.760269,0.458559,-9.609082,-0.762471,-2.497308,-5.470651,-1.837863,-1.201724,0.421541,0.432708,-0.768696,-0.007733,-0.445317,0.080840,-0.098990,0.731948,0.363058,2.965264


In [154]:
y_train.value_counts()

0    219064
1       349
Name: Class, dtype: int64

In [155]:
y_train_sm.value_counts()

0    219064
1    219064
Name: Class, dtype: int64

In [156]:
X_train.shape

(219413, 30)

In [157]:
X_train_sm.shape

(438128, 30)

In [158]:
y_train.shape

(219413,)

In [159]:
y_train_sm.shape

(438128,)

In [160]:
X_test.shape

(54854, 30)

In [161]:
y_test.shape

(54854,)

In [162]:
y_test.value_counts()

0    54764
1       90
Name: Class, dtype: int64

# Evaluate Models

## Random Forest after SMOTE

In [163]:
#random_forest = RandomForestClassifier(min_samples_leaf=100,n_estimators=50)
random_forest = RandomForestClassifier()
random_forest.fit(X_train_sm,y_train_sm)

RandomForestClassifier()

In [164]:
y_prob=random_forest.predict_proba(X_test)
y_pred=random_forest.predict(X_test)

### Evaluating Performance

In [167]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, confusion_matrix

In [169]:
rfc_cv_score = cross_val_score(random_forest, X, y, cv=10, scoring='roc_auc')

In [170]:
print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))
print('\n')
print("=== Classification Report ===")
print(classification_report(y_test, y_pred))
print('\n')
print("=== All AUC Scores ===")
print(rfc_cv_score)
print('\n')
print("=== Mean AUC Score ===")
print("Mean AUC Score - Random Forest: ", rfc_cv_score.mean())

=== Confusion Matrix ===
[[54754    10]
 [    9    81]]


=== Classification Report ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     54764
           1       0.89      0.90      0.90        90

    accuracy                           1.00     54854
   macro avg       0.94      0.95      0.95     54854
weighted avg       1.00      1.00      1.00     54854



=== All AUC Scores ===
[0.92365411 0.97409557 0.99999502 0.85934953 0.90592662 0.89844354
 0.98816826 0.95199661 0.96137144 0.90709059]


=== Mean AUC Score ===
Mean AUC Score - Random Forest:  0.9370091293856075


In [171]:
#print('Confusion Matrix')
#print('='*18)
#print(confusion_matrix(y_test,y_pred),"\n")
#print('Classification Report')
#print('='*22)
#print(classification_report(y_test,y_pred),"\n")
#print('AUC-ROC')
#print('='*8)
#print(roc_auc_score(y_test, y_prob[:,1]))

In [172]:
#fpr, tpr, thresholds = roc_curve(y_test, random_forest.predict_proba(X_test)[:,1],
#                                         pos_label=1)
#plt.title('Receiver Operating Characteristic')
#plt.plot(fpr, tpr, 'b',
#label='AUC = %0.2f'% roc_auc_score(y_test,random_forest.predict_proba(X_test)[:,1]))
#plt.legend(loc='lower right')
#plt.plot([0,1],[0,1],'r--')
#plt.xlim([0,1])
#plt.ylim([0,1])
#plt.ylabel('True Positive Rate')
#plt.xlabel('False Positive Rate')
#plt.show()

### Tuning Hyperparameters

In [178]:
from sklearn.model_selection import RandomizedSearchCV
# number of trees in random forest
n_estimators = [int(x) for x in np.linspace(start = 200, stop = 1000, num = 100)]
# number of features at every split
max_features = ['sqrt']

# max depth
max_depth = [int(x) for x in np.linspace(100, 500, num = 10)]
max_depth.append(None)
# create random grid
random_grid = {
 'n_estimators': n_estimators,
 'max_features': max_features,
 'max_depth': max_depth
 }
# Random search of parameters
rfc_random = RandomizedSearchCV(estimator = random_forest, param_distributions = random_grid, n_iter = 100, cv = 3, verbose=2, random_state=42, n_jobs = -1)
# Fit the model
rfc_random.fit(X_train_sm, y_train_sm)
# print results
print(rfc_random.best_params_)

Fitting 3 folds for each of 100 candidates, totalling 300 fits


KeyboardInterrupt: 

In [ ]:
rfc = RandomForestClassifier(n_estimators=600, max_depth=300, max_features='sqrt')
rfc.fit(X_train_sm,y_train_sm)
rfc_predict = rfc.predict(X_test)
rfc_cv_score = cross_val_score(rfc, X, y, cv=10, scoring='roc_auc')
print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, rfc_predict))
print('\n')
print("=== Classification Report ===")
print(classification_report(y_test, rfc_predict))
print('\n')
print("=== All AUC Scores ===")
print(rfc_cv_score)
print('\n')
print("=== Mean AUC Score ===")
print("Mean AUC Score - Random Forest: ", rfc_cv_score.mean())

In [132]:
#rf_mod = RandomForestClassifier(max_features = 'sqrt', random_state=886)

In [133]:
#rf_mod.fit(X_train_sm,y_train_sm)

RandomForestClassifier(random_state=886)

In [134]:
#print("Random Forest Train R2: ", rf_mod.score(X_train, y_train))
#print("Random Forest Test R2: ", rf_mod.score(X_test, y_test))

Random Forest Train R2:  1.0
Random Forest Test R2:  0.9996353957778831


In [135]:
#rf_mod.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': None,
 'max_features': 'sqrt',
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 886,
 'verbose': 0,
 'warm_start': False}

In [139]:
#param_grid = [
#  {'max_features': [1, 2]} 
#]

In [137]:
#cv_grid_rf = GridSearchCV(rf_mod, param_grid, scoring="r2")

In [138]:
#cv_grid_rf.fit(X_train_sm, y_train_sm)

KeyboardInterrupt: 

In [48]:
#pd.DataFrame(cv_grid_rf.cv_results_)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_max_features,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,38.752281,0.249923,0.725637,0.014062,1,{'max_features': 1},0.999772,0.999726,0.999498,0.999635,0.999452,0.999617,0.000125,2
1,71.122399,1.762164,0.657380,0.025779,2,{'max_features': 2},0.999863,0.999680,0.999543,0.999635,0.999498,0.999644,0.000127,1


In [50]:
#rf_mod_cvfinal = RandomForestClassifier(max_features=2, random_state=886)
#rf_mod_cvfinal.fit(X_train_sm, y_train_sm)

RandomForestClassifier(max_features=2, random_state=886)

In [51]:
#print("Random Forest with CV train R2: ", rf_mod_cvfinal.score(X_train, y_train))
#print("Random Forest with CV test R2: ", rf_mod_cvfinal.score(X_test, y_test))

Random Forest with CV train R2:  1.0
Random Forest with CV test R2:  0.9994713238779305


In [52]:
## Prediction
#y_pred=rf_mod_cvfinal.predict(X_test)

In [53]:
### Check Accuracy
#from sklearn.metrics import accuracy_score
#score=accuracy_score(y_test,y_pred)

In [54]:
#score

0.9994713238779305

In [55]:
### Create a Pickle file using serialization 
import pickle
pickle_out = open("classifier.pkl","wb")
pickle.dump(, pickle_out)
pickle_out.close()

In [62]:
rf_mod_cvfinal.predict([[1,
-3.0435406239976,
-3.15730712090228,
1.08846277997285,
2.2886436183814,
1.35980512966107,
-1.06482252298131,
0.325574266158614,
-0.0677936531906277,
-0.270952836226548,
-0.838586564582682,
-0.414575448285725,
-0.503140859566824,
0.676501544635863,
-1.69202893305906,
2.00063483909015,
0.666779695901966,
0.599717413841732,
1.72532100745514,
0.283344830149495,
2.10233879259444,
0.661695924845707,
0.435477208966341,
1.37596574254306,
-0.293803152734021,
0.279798031841214,
-0.145361714815161,
-0.252773122530705,
0.0357642251788156,
529000000000,]])

array([0])

In [65]:
importances = rf_mod_cvfinal.feature_importances_
sorted_indices = np.argsort(importances)[::-1]
 
feat_labels = X_train_sm.columns[0:]
 
for f in range(X_train_sm.shape[1]):
    print("%2d) %-*s %f" % (f + 1, 50,
                            feat_labels[sorted_indices[f]],
                            importances[sorted_indices[f]]))

AttributeError: 'numpy.ndarray' object has no attribute 'columns'